In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

from sklearn.ensemble import RandomForestRegressor

from src.data_loaders import load_order_profit_raw

ModuleNotFoundError: No module named 'src'

In [3]:
df = load_order_profit_raw()

df.shape, df.columns

NameError: name 'load_order_profit_raw' is not defined

In [ ]:
target = "net_operating_profit"

In [ ]:
df[target].describe()

In [ ]:
numeric_features = [
    "discount_pct",
    "shipping_fee_charged",
    "actual_shipping_cost",
    "return_handling_cost",
    "support_cost",
    "marketing_cost_per_order",
    "gross_profit",
    "net_revenue",
]

categorical_features = [
    "discount_band",
    "acquisition_channel",
    "state_region",
    "segment",
]

In [ ]:
numeric_transformer = "passthrough"

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [ ]:
ridge = Ridge(alpha=1.0)

ridge_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", ridge)
    ]
)

X = df[numeric_features + categorical_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ridge_model.fit(X_train, y_train)
preds = ridge_model.predict(X_test)

r2_score(y_test, preds)

In [ ]:
ohe = ridge_model.named_steps["preprocessor"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(categorical_features)

feature_names = np.concatenate([numeric_features, cat_names])

coeffs = ridge_model.named_steps["model"].coef_

ridge_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": coeffs
})

ridge_importance["abs_importance"] = ridge_importance["importance"].abs()
ridge_importance = ridge_importance.sort_values("abs_importance", ascending=False)

ridge_importance.head(20)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=10,
    n_jobs=-1
)

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf)
    ]
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

r2_score(y_test, rf_preds)

In [ ]:
rf_importance = rf_model.named_steps["model"].feature_importances_

rf_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_importance
}).sort_values("importance", ascending=False)

rf_df.head(20)

In [ ]:
combined = ridge_importance[["feature", "importance"]].rename(
    columns={"importance": "ridge_importance"}
)

rf_trim = rf_df[["feature", "importance"]].rename(
    columns={"importance": "rf_importance"}
)

driver_importance = combined.merge(rf_trim, on="feature", how="outer")

driver_importance = driver_importance.fillna(0)

driver_importance["combined_score"] = (
    driver_importance["ridge_importance"].abs() +
    driver_importance["rf_importance"]
)

driver_importance = driver_importance.sort_values(
    "combined_score",
    ascending=False
)

driver_importance.head(25)

In [ ]:
driver_importance.to_csv(
    "data/driver_importance.csv",
    index=False
)

In [ ]:
top = driver_importance.head(10)
top